# Binary Autoencoder Symbolic Train
This notebook trains or loads the BinaryConv autoencoder, then exports symbolic feature/predicate CSVs for ASP evaluation. It includes an adaptive reservoir variant that actually changes the latent symbolic state before feature extraction.


## Training + Symbolic Features

- Stage 1: Binary Convolutional Autoencoder with STE Thresholding produces symbolic latents.
- Stage 2: Optional adaptive signed reservoir transforms the latent state.
- Stage 3: Symbolic features are derived from the transformed latents, reservoir transitions, lesion masks, and color statistics.
- Stage 4: ASP evaluation is run in `Binary_Autoencoder_Symbolic_Evaluation.ipynb`.


In [1]:
# Imports and config
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Sequence, Any
import random
import json as pyjson
import os

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF



def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@dataclass
class Config:
    ckpt_prefix: str = 'binaryconvautoencoder-v4'
    multitask_dataset: bool = True

    project_root: Path = Path.cwd()
    dataset_root: Path = Path(os.getcwd()) / ".." / "Medical_Dataset" / "Full_Labels" / "multitask_fully_labeled"
    manifest_csv: Path = dataset_root / 'manifest.csv'
    image_size: Tuple[int, int] = (256, 256)
    batch_size: int = 16

    max_epochs_ae: int = 30
    base_learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    early_stop_patience: int = 5
    early_stop_min_delta: float = 1e-4

    class_names: Tuple[str, ...] = ('MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC')
    malignant_classes: Tuple[str, ...] = ('MEL', 'BCC', 'AKIEC')
    use_binary_labels: bool = True

    latent_channels: int = 64
    binconv_depth: int = 3

    # Reservoir export defaults. Use the same source autoencoder checkpoint for
    # ReservoirAdaptive and NoReservoirAdaptive so the ablation isolates stage 2.
    run_seed: int = 42
    variant_name: str = 'NoReservoir'
    source_model_variant_name: str = 'NoReservoir'
    use_reservoir: bool = False
    reservoir_layers: int = 3
    reservoir_steps: int = 4
    reservoir_target_density: float = 0.15
    reservoir_state_coupling: float = 0.90
    reservoir_input_coupling: float = 0.50
    reservoir_leak: float = 0.25
    reservoir_bias: float = 0.0
    reservoir_until_converged: bool = False
    reservoir_stable_tol: float = 0.0

    ae_mask_loss_weight: float = 0.2
    ae_density_weight: float = 0.05
    ae_target_density: float = 0.08
    ae_aux_cls_weight: float = 0.2
    ae_aux_hidden: int = 128

    base_out_dir: Path = Path(os.getcwd()) / "Results"
    out_dir: Path = Path()
    in_models_dir: Path = Path()
    models_dir: Path = Path()

    predicate_quantiles: Tuple[float, float, float, float, float] = (0.1, 0.3, 0.5, 0.7, 0.9)
    predicate_min_rate: float = 0.01
    predicate_max_rate: float = 0.99

    use_symbolic_feedback: bool = True
    symbolic_feedback_weight: float = 0.2
    feedback_train_csv: Path = Path()
    feedback_val_csv: Path = Path()

    skip_training: bool = False

    in_ckpt_best_full: Path = Path()
    in_ckpt_best_partial: Path = Path()
    in_ckpt_last_full: Path = Path()
    in_ckpt_best_or_last_path: Path = Path()
    out_ckpt_best: Path = Path()
    out_ckpt_last: Path = Path()
    out_ckpt_best_partial: Path = Path()
    models_history: Path = Path()
    symbol_out_dir: Path = Path()


cfg = Config()

set_seed(cfg.run_seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Seed used:', cfg.run_seed)


Using device: cuda
Seed used: 42


In [2]:
cfg.skip_training = True

In [3]:
# Environment detection: local vs Kaggle
import os
from pathlib import Path


def detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or Path('/kaggle/input').exists():
        return 'kaggle'
    return 'local'


ENV = detect_env()
print('Environment:', ENV)


def _result_dir(base_dir: Path, variant_name: str, seed: int) -> Path:
    return base_dir / variant_name / f"results_seed{seed}"


def _resolve_local_paths() -> tuple[Path, Path, Path]:
    cwd = Path(os.getcwd()).resolve()
    dataset_rel = Path('Medical_Dataset') / 'Full_Labels' / 'multitask_fully_labeled'

    if (cwd / dataset_rel).exists():
        project_root = cwd
        github_root = cwd / 'Github' if (cwd / 'Github').exists() else cwd
    elif (cwd.parent / dataset_rel).exists():
        project_root = cwd.parent
        github_root = cwd
    else:
        project_root = cwd.parent
        github_root = cwd

    dataset_root = project_root / dataset_rel
    base_out_dir = github_root / 'Results'
    return project_root, dataset_root, base_out_dir


def set_dirs(env_name: str):
    if env_name == 'kaggle':
        cfg.project_root = Path('/kaggle/working')
        cfg.dataset_root = Path('/kaggle/input/datasets/enricotazzer/multitask-dataset-pseudo-labeled/multitask_fully_labeled')
        cfg.base_out_dir = Path('/kaggle/working/Results')

        kaggle_model_dir = Path(f'/kaggle/input/models/luca45/{cfg.ckpt_prefix}-seed{cfg.run_seed}/pytorch/default/1')
        local_style_source = _result_dir(cfg.base_out_dir, cfg.source_model_variant_name, cfg.run_seed) / 'Models'
        cfg.in_models_dir = kaggle_model_dir if kaggle_model_dir.exists() else local_style_source
    else:
        cfg.project_root, cfg.dataset_root, cfg.base_out_dir = _resolve_local_paths()
        cfg.in_models_dir = _result_dir(cfg.base_out_dir, cfg.source_model_variant_name, cfg.run_seed) / 'Models'

    cfg.manifest_csv = cfg.dataset_root / 'manifest.csv'
    cfg.out_dir = _result_dir(cfg.base_out_dir, cfg.variant_name, cfg.run_seed)
    cfg.models_dir = cfg.out_dir / 'Models'
    cfg.symbol_out_dir = cfg.out_dir / f'Seed{cfg.run_seed}_Feat'
    cfg.feedback_train_csv = cfg.out_dir / 'asp_feedback_train.csv'
    cfg.feedback_val_csv = cfg.out_dir / 'asp_feedback_val.csv'

    cfg.in_ckpt_best_full = cfg.in_models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_best_full.pth'
    cfg.in_ckpt_best_partial = cfg.in_models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_best.pth'
    cfg.in_ckpt_last_full = cfg.in_models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_last_full.pth'
    cfg.in_ckpt_best_or_last_path = cfg.in_ckpt_best_full if cfg.in_ckpt_best_full.exists() else cfg.in_ckpt_last_full

    cfg.out_ckpt_best = cfg.models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_best_full.pth'
    cfg.out_ckpt_last = cfg.models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_last_full.pth'
    cfg.out_ckpt_best_partial = cfg.models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_best.pth'
    cfg.models_history = cfg.models_dir / f'{cfg.ckpt_prefix}_seed{cfg.run_seed}_history.json'


def mkdirs():
    cfg.out_dir.mkdir(parents=True, exist_ok=True)
    cfg.models_dir.mkdir(parents=True, exist_ok=True)
    cfg.symbol_out_dir.mkdir(parents=True, exist_ok=True)


set_dirs(ENV)

assert cfg.dataset_root.exists(), f"Dataset not found at {cfg.dataset_root}"
mkdirs()
assert cfg.out_dir.exists(), f"Output dir is not found at {cfg.out_dir}"
assert cfg.models_dir.exists(), f"Models dir is not found at {cfg.models_dir}"
assert cfg.symbol_out_dir.exists(), f"Symbolic dir is not found at {cfg.symbol_out_dir}"

if cfg.skip_training:
    has_source_ckpt = cfg.in_ckpt_best_partial.exists() or cfg.in_ckpt_best_full.exists() or cfg.in_ckpt_last_full.exists()
    assert has_source_ckpt, f"No source checkpoint found in {cfg.in_models_dir}"

print('Project root:', cfg.project_root)
print('Source model dir:', cfg.in_models_dir)
print('Output dir:', cfg.out_dir)
print('Symbolic output dir:', cfg.symbol_out_dir)


Environment: local
Project root: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico
Source model dir: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Models
Output dir: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42
Symbolic output dir: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Seed42_Feat


In [4]:
# Data pipeline (RGB images + masks)

def load_image_rgb(path: Path, size, train=False):
    
    img = Image.open(path).convert('RGB')
    if train:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if random.random() < 0.2:
            img = TF.vflip(img)
        if random.random() < 0.3:
            angle = random.uniform(-15, 15)
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
    img = TF.resize(img, size, interpolation=InterpolationMode.BILINEAR)
    img = TF.to_tensor(img)
    return img


def load_mask(path: Path, size, train=False):
    mask = Image.open(path).convert('L')
    if train:
        if random.random() < 0.5:
            mask = TF.hflip(mask)
        if random.random() < 0.2:
            mask = TF.vflip(mask)
        if random.random() < 0.3:
            angle = random.uniform(-15, 15)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
    mask = TF.resize(mask, size, interpolation=InterpolationMode.NEAREST)
    mask = TF.to_tensor(mask)
    mask = (mask > 0.5).float()
    return mask


class ImageDataset(Dataset):
    def __init__(self, cfg: Config, split: str):
        self.cfg = cfg
        self.split = split
        self.df = pd.read_csv(cfg.manifest_csv)
        self.df = self.df[self.df['split'] == split].reset_index(drop=True)
        if self.df.empty:
            raise RuntimeError(f'No rows for split={split}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.cfg.dataset_root / row['image']
        mask_path = self.cfg.dataset_root / row['mask']
        img = load_image_rgb(img_path, self.cfg.image_size, train=(self.split=='train'))
        mask = load_mask(mask_path, self.cfg.image_size, train=(self.split=='train'))
        label_vec = row[list(self.cfg.class_names)].to_numpy(dtype=np.float32)
        class_id = int(label_vec.argmax())
        class_name = self.cfg.class_names[class_id]
        label_bin = int(class_name in self.cfg.malignant_classes)
        return {
            'image': img,
            'mask': mask,
            'class_id': torch.tensor(class_id, dtype=torch.long),
            'label_bin': torch.tensor(label_bin, dtype=torch.long),
            'image_id': row['image_id'],
            'image_path': str(img_path),
            'mask_path': str(mask_path),
        }


def create_loaders(cfg: Config):
    datasets = {split: ImageDataset(cfg, split) for split in ('train', 'val', 'test')}
    loaders = {
        split: DataLoader(datasets[split], batch_size=cfg.batch_size, shuffle=(split=='train'), drop_last=(split=='train'))
        for split in ('train', 'val', 'test')
    }
    return datasets, loaders


datasets, loaders = create_loaders(cfg)
print({k: len(v) for k, v in datasets.items()})


{'train': 10790, 'val': 2312, 'test': 2312}


In [5]:
# BinaryConv core (Stage 1 - Symbolic encoding)
class STEBinarize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return (x > 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output


def ste_binarize(x: torch.Tensor, threshold: float = 0.0) -> torch.Tensor:
    hard = (x >= threshold).float()
    return x + (hard - x).detach()


class BinaryConvCell(nn.Module):
    # Binary, local-rule cellular update (3x3 neighborhood, STE binarization).
    def __init__(self, in_channels: int, out_channels: int, steps: int = 1,
                 alpha: float = 1.0, beta: float = 0.0,
                 trainable: bool = False, learnable_gain: bool = False, learnable_bias: bool = False,
                 detach_dynamics: bool = True):
        super().__init__()
        self.steps = max(1, steps)
        self.detach_dynamics = detach_dynamics
        self.alpha = nn.Parameter(torch.tensor(alpha), requires_grad=learnable_gain)
        self.beta = nn.Parameter(torch.tensor(beta), requires_grad=learnable_bias)
        self.template_state = nn.Parameter(torch.randn(out_channels, in_channels, 3, 3) * 0.1, requires_grad=trainable)
        self.template_input = nn.Parameter(torch.randn(out_channels, in_channels, 3, 3) * 0.1, requires_grad=trainable)
        if not trainable:
            for param in [self.template_state, self.template_input]:
                param.requires_grad = False

    def freeze(self):
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        driven = F.conv2d(x, self.template_input, padding=1)
        if self.detach_dynamics:
            driven = driven.detach()
        state = ste_binarize(driven)
        for _ in range(self.steps):
            s_in = state.detach() if self.detach_dynamics else state
            neighborhood = F.conv2d(s_in, self.template_state, padding=1)
            combined = self.alpha * s_in + neighborhood + driven + self.beta
            state = ste_binarize(combined)
        return state


class BinaryConvAutoencoder(nn.Module):
    def __init__(self, latent_channels: int = 64, depth: int = 3,
                 trainable_input_lift: bool = False,
                 trainable_templates: bool = False, learnable_gain: bool = False, learnable_bias: bool = False):
        super().__init__()
        self.trainable_input_lift = trainable_input_lift
        self.latent_requires_grad = trainable_input_lift or trainable_templates or learnable_gain or learnable_bias
        self.input_lift = nn.Conv2d(3, latent_channels, kernel_size=3, padding=1, bias=False)
        if not self.trainable_input_lift:
            for param in self.input_lift.parameters():
                param.requires_grad = False
        self.encoder_cells = nn.ModuleList([
            BinaryConvCell(latent_channels, latent_channels, steps=2,
                       trainable=trainable_templates,
                       learnable_gain=learnable_gain,
                       learnable_bias=learnable_bias)
            for _ in range(depth)
        ])
        self.decoder_cells = nn.ModuleList([
            BinaryConvCell(latent_channels, latent_channels, steps=2,
                       trainable=trainable_templates,
                       learnable_gain=learnable_gain,
                       learnable_bias=learnable_bias)
            for _ in range(depth)
        ])
        self.output_head = nn.Conv2d(latent_channels, 3, kernel_size=3, padding=1, bias=True)

    def freeze_dynamics(self) -> None:
        for param in self.input_lift.parameters():
            param.requires_grad = False
        for cell in list(self.encoder_cells) + list(self.decoder_cells):
            cell.freeze()
        for param in self.output_head.parameters():
            param.requires_grad = False
        self.latent_requires_grad = False

    def trainable_parameters(self):
        if self.trainable_input_lift:
            yield from self.input_lift.parameters()
        for cell in list(self.encoder_cells) + list(self.decoder_cells):
            for p in cell.parameters():
                if p.requires_grad:
                    yield p
        for p in self.output_head.parameters():
            if p.requires_grad:
                yield p

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        z = self.input_lift(x)
        for cell in self.encoder_cells:
            z = cell(z)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        x = z
        for cell in self.decoder_cells:
            x = cell(x)
        return self.output_head(x)

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        recon = self.decode(z)
        return recon, z


In [6]:
# Stage 2: Adaptive signed BinaryConv Reservoir

def _rank_tie_breaker_like(x: torch.Tensor) -> torch.Tensor:
    if x.dim() != 4:
        raise ValueError(f'Expected BCHW tensor, got shape={tuple(x.shape)}')
    _, channels, height, width = x.shape
    c = torch.arange(channels, device=x.device, dtype=x.dtype).view(1, channels, 1, 1)
    y = torch.arange(height, device=x.device, dtype=x.dtype).view(1, 1, height, 1)
    w = torch.arange(width, device=x.device, dtype=x.dtype).view(1, 1, 1, width)
    return torch.remainder((c + 1) * 0.61803398875 + (y + 1) * 0.41421356237 + (w + 1) * 0.73205080757, 1.0)


def adaptive_binarize(x: torch.Tensor, target_density: float = 0.15, tie_epsilon: float = 1e-4) -> torch.Tensor:
    # Binarize by per-sample rank so saturated latents still become sparse symbolic states.
    added_batch = False
    work = x
    if work.dim() == 3:
        work = work.unsqueeze(0)
        added_batch = True
    if work.dim() != 4:
        raise ValueError(f'Expected CHW or BCHW tensor, got shape={tuple(x.shape)}')

    density = float(np.clip(target_density, 1e-4, 0.9999))
    scores = work + tie_epsilon * _rank_tie_breaker_like(work)
    flat = scores.flatten(1)
    threshold = torch.quantile(flat, 1.0 - density, dim=1, keepdim=True)
    hard = (flat >= threshold).reshape_as(work).to(work.dtype)
    out = work + (hard - work).detach() if work.requires_grad else hard
    return out.squeeze(0) if added_batch else out


class BinaryConvReservoir(nn.Module):
    def __init__(
        self,
        channels: int = 64,
        depth: int = 3,
        steps: int = 4,
        target_density: float = 0.15,
        state_coupling: float = 0.90,
        input_coupling: float = 0.50,
        leak: float = 0.25,
        bias: float = 0.0,
        memory_mode: str = "replace",
        memory_decay: float = 0.25,
        inject_input_each_step: bool = True,
        until_converged: bool = False,
        stable_tol: float = 0.0,
        max_converge_steps: int | None = None,
    ) -> None:
        super().__init__()
        self.channels = int(channels)
        self.depth = max(1, int(depth))
        self.virtual_steps = max(1, int(steps))
        self.target_density = float(target_density)
        self.state_coupling = float(state_coupling)
        self.input_coupling = float(input_coupling)
        self.leak = float(leak)
        self.bias = float(bias)
        self.memory_mode = memory_mode
        self.memory_decay = float(memory_decay)
        self.inject_input_each_step = bool(inject_input_each_step)
        self.until_converged = bool(until_converged)
        self.stable_tol = float(stable_tol)
        self.max_converge_steps = max_converge_steps

        state_base = torch.tensor([
            [-0.75,  1.00, -0.75],
            [ 1.00,  0.00,  1.00],
            [-0.75,  1.00, -0.75],
        ], dtype=torch.float32)
        input_base = torch.tensor([
            [0.25, 0.50, 0.25],
            [0.50, 0.00, 0.50],
            [0.25, 0.50, 0.25],
        ], dtype=torch.float32)
        state_base = state_base / (state_base.abs().sum() + 1e-6)
        input_base = input_base / (input_base.abs().sum() + 1e-6)

        state_kernel = torch.zeros(self.depth, self.channels, 1, 3, 3)
        input_kernel = torch.zeros(self.depth, self.channels, 1, 3, 3)
        state_kernel[:, :, 0] = self.state_coupling * state_base
        input_kernel[:, :, 0] = self.input_coupling * input_base
        self.register_buffer('state_kernel', state_kernel, persistent=False)
        self.register_buffer('input_kernel', input_kernel, persistent=False)

        self._states: list[torch.Tensor] | None = None

    def reset_state(self) -> None:
        self._states = None

    def _ensure_states(self, initial_state: torch.Tensor) -> None:
        if self._states is None or len(self._states) != self.depth or self._states[0].shape != initial_state.shape:
            self._states = [initial_state.clone() for _ in range(self.depth)]

    @torch.no_grad()
    def forward(
        self,
        latent: torch.Tensor,
        steps: int | None = None,
        reset_state: bool = False,
        until_converged: bool | None = None,
        return_diagnostics: bool = False,
    ):
        if reset_state:
            self.reset_state()

        input_state = adaptive_binarize(latent.detach(), self.target_density)
        self._ensure_states(input_state)

        k = max(1, int(steps or self.virtual_steps))
        stop_on_stable = self.until_converged if until_converged is None else until_converged
        max_iters = self.max_converge_steps or k
        delta_history: list[torch.Tensor] = []
        density_history: list[torch.Tensor] = []

        for it in range(max_iters):
            drive = input_state
            new_states: list[torch.Tensor] = []
            layer_deltas: list[torch.Tensor] = []
            for layer_idx in range(self.depth):
                state = self._states[layer_idx]
                external_source = drive if self.inject_input_each_step else input_state
                recurrent = F.conv2d(state, self.state_kernel[layer_idx], padding=1, groups=self.channels)
                external = F.conv2d(external_source, self.input_kernel[layer_idx], padding=1, groups=self.channels)
                pre_activation = self.leak * state + recurrent + external + self.bias
                updated = adaptive_binarize(pre_activation, self.target_density)

                if self.memory_mode == 'or':
                    merged = torch.maximum(state, updated)
                elif self.memory_mode == 'ema':
                    merged = adaptive_binarize(
                        self.memory_decay * state + (1.0 - self.memory_decay) * updated,
                        self.target_density,
                    )
                else:
                    merged = updated

                new_states.append(merged)
                layer_deltas.append((merged - state).abs().mean(dim=(1, 2, 3)))
                drive = merged

            delta = torch.stack(layer_deltas, dim=0).mean(dim=0)
            self._states = new_states
            delta_history.append(delta.detach().cpu())
            density_history.append(self._states[-1].mean(dim=(1, 2, 3)).detach().cpu())

            if stop_on_stable and delta.max().item() <= self.stable_tol:
                break
            if not stop_on_stable and it + 1 >= k:
                break

        final_state = self._states[-1].clone()
        if not return_diagnostics:
            return final_state

        diagnostics = {
            'delta_history': torch.stack(delta_history, dim=1) if delta_history else torch.zeros(latent.size(0), 0),
            'density_history': torch.stack(density_history, dim=1) if density_history else torch.zeros(latent.size(0), 0),
            'steps_run': len(delta_history),
        }
        return final_state, diagnostics


In [7]:
# Train BinaryConv autoencoder
model = BinaryConvAutoencoder(
    latent_channels=cfg.latent_channels,
    depth=cfg.binconv_depth,
    trainable_input_lift=True,
    trainable_templates=True,
    learnable_gain=True,
    learnable_bias=True,
)

if not cfg.skip_training:
    model = train_autoencoder(model, loaders, cfg.max_epochs_ae, cfg.ckpt_prefix)
else:
    ckpt = cfg.in_ckpt_best_partial if cfg.in_ckpt_best_partial.exists() else cfg.in_ckpt_best_or_last_path
    print('Training skipped; checkpoint selected:', ckpt)


Training skipped; checkpoint selected: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Models\binaryconvautoencoder-v4_seed42_best.pth


In [8]:
AE_LOSS_OUT_FILE = cfg.models_dir / f"AE_Recon_Loss_seed{cfg.run_seed}.json"

if cfg.skip_training:
    model = model.to(device)
    raw_ckpt = torch.load(ckpt, map_location=device)
    if isinstance(raw_ckpt, dict) and 'model_state_dict' in raw_ckpt:
        model.load_state_dict(raw_ckpt['model_state_dict'])
    else:
        model.load_state_dict(raw_ckpt)
    print('Train skipped. Loaded checkpoint:', ckpt)


def eval_recon(model, loader):
    model.eval()
    loss_fn = nn.MSELoss()
    total = 0.0
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            recon, _ = model(images)
            total += loss_fn(recon, images).item()
    return total / max(1, len(loader))


recon_loss_train = eval_recon(model, loaders['train'])
recon_loss_test = eval_recon(model, loaders['test'])

print('AE test recon loss:', recon_loss_test)
print('Results saved in:', AE_LOSS_OUT_FILE)

report = {
    cfg.run_seed: {
        'source_checkpoint': str(ckpt) if cfg.skip_training else str(cfg.out_ckpt_best),
        'recon_loss_train': recon_loss_train,
        'recon_loss_test': recon_loss_test,
    }
}

with AE_LOSS_OUT_FILE.open('w', encoding='utf-8') as f:
    pyjson.dump(report, f, indent=2)


Train skipped. Loaded checkpoint: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Models\binaryconvautoencoder-v4_seed42_best.pth
AE test recon loss: 0.015392264145715484
Results saved in: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Models\AE_Recon_Loss_seed42.json


In [9]:
# Extract symbolic features with reservoir + masks and save CSVs
# NOTE: Updated with richer predicates + quantile binning.


# --- Color features (expanded, HSV bins + entropy) ---

def _load_rgb_uint8(path: str) -> np.ndarray:
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def _load_mask_bool(path: Optional[str]) -> Optional[np.ndarray]:
    if not path:
        return None
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    mask_bool = mask > 127
    if not mask_bool.any():
        return None
    return mask_bool


def compute_hsv_color_concepts(image_path: str, mask_path: Optional[str] = None, hue_bins: int = 16) -> Dict[str, float]:  #V4_Change more hue bins
    rgb = _load_rgb_uint8(image_path)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    lesion_mask = _load_mask_bool(mask_path)
    if lesion_mask is not None:
        pixels = hsv[lesion_mask]
    else:
        pixels = hsv.reshape(-1, 3)
    if pixels.size == 0:
        pixels = hsv.reshape(-1, 3)
    h = pixels[:, 0] / 180.0
    s = pixels[:, 1] / 255.0
    v = pixels[:, 2] / 255.0
    hue_edges = np.linspace(0.0, 1.0, hue_bins + 1)
    hue_hist, _ = np.histogram(h, bins=hue_edges)
    hue_hist = hue_hist.astype(np.float32) / (hue_hist.sum() + 1e-6)

    hue_variety = float((hue_hist > 0.08).sum())  #V4_Change: lower threshold for richer signals
    saturation_mean = float(s.mean())
    value_mean = float(v.mean())
    red_ratio = float(((h <= 0.08) | (h >= 0.92)).mean())

    blue_ratio = float(((h >= 0.55) & (h <= 0.70)).mean())
    saturation_std = float(s.std())
    value_std = float(v.std())
    hue_entropy = float(-np.sum(hue_hist * np.log(hue_hist + 1e-6)))

    stats = {
        'value_mean': value_mean,
        'saturation_mean': saturation_mean,
        'hue_variety': hue_variety,
        'red_ratio': red_ratio,
        'blue_ratio': blue_ratio,
        'saturation_std': saturation_std,
        'value_std': value_std,
        'hue_entropy': hue_entropy,
    }

    # expose raw hue-bin proportions (numeric, will be quantile-binned)
    for b in range(hue_bins):
        stats[f'hue_bin_{b}'] = float(hue_hist[b])

    stats['color_dark'] = value_mean < 0.45
    stats['color_variegated'] = hue_variety >= 2
    stats['color_reddish'] = red_ratio >= 0.15
    stats['color_bluish'] = blue_ratio >= 0.05
    stats['color_irregular'] = saturation_std >= 0.15 or value_std >= 0.15

    return stats


# --- Latent features (expanded shape/texture) [CHANGE] ---

def compute_latent_features(latent: torch.Tensor, mask: torch.Tensor) -> Dict[str, float]:
    latent_np = latent.detach().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()

    binary = (latent_np > 0.5).astype(np.float32)
    num_channels, h, w = binary.shape
    total_pixels = float(h * w)

    latent_density = float(binary.mean())
    channel_activations = binary.mean(axis=(1, 2))
    area_ratio = float(channel_activations.var())

    mean_activation_map = binary.mean(axis=0)

    # symmetry (simple)
    if mean_activation_map.sum() > 0:
        horiz = np.abs(mean_activation_map - np.fliplr(mean_activation_map)).mean()
        vert = np.abs(mean_activation_map - np.flipud(mean_activation_map)).mean()
        symmetry_score = float(1.0 - 0.5 * (horiz + vert))
    else:
        symmetry_score = 0.0

    # compactness
    if mean_activation_map.sum() > 0:
        yy, xx = np.mgrid[:h, :w]
        total_mass = mean_activation_map.sum()
        cy = (yy * mean_activation_map).sum() / total_mass
        cx = (xx * mean_activation_map).sum() / total_mass
        distances = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
        mean_distance = (distances * mean_activation_map).sum() / total_mass
        max_possible_distance = np.sqrt(h ** 2 + w ** 2) / 2
        compactness = 1.0 - (mean_distance / max_possible_distance)
    else:
        compactness = 0.0

    # channel diversity (entropy)
    probs = channel_activations / (channel_activations.sum() + 1e-6)
    probs = probs[probs > 1e-6]
    if probs.size > 0:
        entropy = -np.sum(probs * np.log(probs + 1e-6))
        max_entropy = np.log(num_channels)
        channel_diversity = float(entropy / max_entropy) if max_entropy > 0 else 0.0
    else:
        channel_diversity = 0.0

    # center-periphery contrast (simple gradient)
    gy, gx = np.gradient(mean_activation_map)
    center_periphery_contrast = float(np.sqrt(gx**2 + gy**2).mean())

    # V4_Change: border irregularity (perimeter/area proxy)
    binary_map = (mean_activation_map > 0.3).astype(np.uint8)
    if binary_map.sum() > 0:
        kernel = np.ones((3, 3), np.uint8)
        eroded = cv2.erode(binary_map, kernel, iterations=1)
        border = (binary_map - eroded).clip(min=0)
        perimeter = float(border.sum())
        area = float(binary_map.sum())
        border_irregularity = 1.0 - min(1.0, (4 * np.pi * area) / (perimeter ** 2 + 1e-6)) if perimeter > 0 else 0.0
    else:
        border_irregularity = 0.0

    # V4_Change: texture heterogeneity (patch variance)
    patch = max(2, h // 4)
    patch_vals = []
    for i in range(0, h - patch + 1, patch):
        for j in range(0, w - patch + 1, patch):
            patch_vals.append(mean_activation_map[i:i+patch, j:j+patch].mean())
    texture_heterogeneity = float(np.var(patch_vals)) if patch_vals else 0.0

    # region stats from mask
    mask_bin = (mask_np > 0.5).astype(np.float32)
    if mask_bin.ndim == 3:
        mask_bin = mask_bin[0]
    region_pixels = mask_bin.sum()
    background_pixels = total_pixels - region_pixels

    region_mean = float((binary * mask_bin).sum() / (region_pixels + 1e-6)) if region_pixels > 0 else 0.0
    background_mean = float((binary * (1 - mask_bin)).sum() / (background_pixels + 1e-6)) if background_pixels > 0 else 0.0
    region_area = float(region_pixels / total_pixels) if total_pixels > 0 else 0.0
    region_bg_contrast = float(region_mean - background_mean)

    if region_pixels > 1:
        region_vals = binary[:, mask_bin.astype(bool)]
        region_coherence = float(1.0 / (region_vals.std() + 1e-6))
    else:
        region_coherence = 0.0

    region_active_channels = float((channel_activations > 0.1).sum())

    #V4_Change: mask compactness (shape of lesion mask)
    mask_u8 = (mask_bin > 0.5).astype(np.uint8)
    if mask_u8.sum() > 0:
        kernel = np.ones((3, 3), np.uint8)
        eroded = cv2.erode(mask_u8, kernel, iterations=1)
        border = (mask_u8 - eroded).clip(min=0)
        per = float(border.sum())
        area = float(mask_u8.sum())
        mask_compactness = min(1.0, (4 * np.pi * area) / (per ** 2 + 1e-6)) if per > 0 else 0.0
    else:
        per = 0.0
        border = np.zeros_like(mask_u8, dtype=np.float32)
        mask_compactness = 0.0

    #V4_Change: mask asymmetry (clinical proxy)
    if mask_u8.sum() > 0:
        m = mask_u8.astype(np.float32)
        asym_h = np.abs(m - np.fliplr(m)).mean()
        asym_v = np.abs(m - np.flipud(m)).mean()
        mask_asymmetry = float(0.5 * (asym_h + asym_v))
    else:
        mask_asymmetry = 0.0

    #V4_Change: mask eccentricity + solidity + border_std
    coords = np.column_stack(np.where(mask_u8 > 0))
    if coords.shape[0] >= 3:
        cov = np.cov(coords.T)
        eigvals = np.linalg.eigvalsh(cov)
        if eigvals.max() > 0:
            mask_eccentricity = float(1.0 - (eigvals.min() / eigvals.max()))
        else:
            mask_eccentricity = 0.0
    else:
        mask_eccentricity = 0.0

    # solidity approximation via convex hull area
    try:
        import cv2 as _cv2
        cnts, _ = _cv2.findContours(mask_u8, _cv2.RETR_EXTERNAL, _cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            cnt = max(cnts, key=_cv2.contourArea)
            hull = _cv2.convexHull(cnt)
            hull_area = float(_cv2.contourArea(hull))
            mask_area = float(_cv2.contourArea(cnt))
            mask_solidity = (mask_area / hull_area) if hull_area > 0 else 0.0
        else:
            mask_solidity = 0.0
    except Exception:
        mask_solidity = 0.0

    # border std as rough irregularity
    if per > 0:
        mask_border_std = float(border.std())
    else:
        mask_border_std = 0.0

    return {
        'latent_density': latent_density,
        'area_ratio': area_ratio,
        'symmetry_score': symmetry_score,
        'compactness': compactness,
        'channel_diversity': channel_diversity,
        'center_periphery_contrast': center_periphery_contrast,
        'border_irregularity': border_irregularity,
        'texture_heterogeneity': texture_heterogeneity,
        'region_mean': region_mean,
        'background_mean': background_mean,
        'region_bg_contrast': region_bg_contrast,
        'region_area': region_area,
        'region_coherence': region_coherence,
        'region_active_channels': region_active_channels,
        'mask_compactness': mask_compactness,
        'mask_eccentricity': mask_eccentricity,
        'mask_solidity': mask_solidity,
        'mask_border_std': mask_border_std,
        'mask_asymmetry': mask_asymmetry,
    }

# --- Reservoir transition features ---

RESERVOIR_FEATURE_COLS = [
    'res_initial_density',
    'res_final_density',
    'res_density_shift',
    'res_changed_fraction',
    'res_on_transition_rate',
    'res_off_transition_rate',
    'res_persistence_rate',
    'res_mean_step_delta',
    'res_last_step_delta',
    'res_mean_step_density',
    'res_step_density_std',
]


def compute_reservoir_transition_features(
    initial_state: torch.Tensor,
    final_state: torch.Tensor,
    diagnostics: Optional[Dict[str, torch.Tensor]] = None,
    row_idx: int = 0,
) -> Dict[str, float]:
    before = (initial_state.detach().cpu().numpy() > 0.5).astype(np.float32)
    after = (final_state.detach().cpu().numpy() > 0.5).astype(np.float32)

    changed = before != after
    on_transition = (before < 0.5) & (after > 0.5)
    off_transition = (before > 0.5) & (after < 0.5)
    persisted_on = (before > 0.5) & (after > 0.5)

    before_on = float(before.sum())
    step_delta = np.array([], dtype=np.float32)
    step_density = np.array([], dtype=np.float32)
    if diagnostics is not None:
        if 'delta_history' in diagnostics and diagnostics['delta_history'].numel() > 0:
            step_delta = diagnostics['delta_history'][row_idx].detach().cpu().numpy().astype(np.float32)
        if 'density_history' in diagnostics and diagnostics['density_history'].numel() > 0:
            step_density = diagnostics['density_history'][row_idx].detach().cpu().numpy().astype(np.float32)

    return {
        'res_initial_density': float(before.mean()),
        'res_final_density': float(after.mean()),
        'res_density_shift': float(after.mean() - before.mean()),
        'res_changed_fraction': float(changed.mean()),
        'res_on_transition_rate': float(on_transition.mean()),
        'res_off_transition_rate': float(off_transition.mean()),
        'res_persistence_rate': float(persisted_on.sum() / (before_on + 1e-6)) if before_on > 0 else 0.0,
        'res_mean_step_delta': float(step_delta.mean()) if step_delta.size else 0.0,
        'res_last_step_delta': float(step_delta[-1]) if step_delta.size else 0.0,
        'res_mean_step_density': float(step_density.mean()) if step_density.size else float(after.mean()),
        'res_step_density_std': float(step_density.std()) if step_density.size else 0.0,
    }


# --- Build feature tables ---

def make_reservoir() -> BinaryConvReservoir:
    return BinaryConvReservoir(
        channels=cfg.latent_channels,
        depth=cfg.reservoir_layers,
        steps=cfg.reservoir_steps,
        target_density=cfg.reservoir_target_density,
        state_coupling=cfg.reservoir_state_coupling,
        input_coupling=cfg.reservoir_input_coupling,
        leak=cfg.reservoir_leak,
        bias=cfg.reservoir_bias,
        until_converged=cfg.reservoir_until_converged,
        stable_tol=cfg.reservoir_stable_tol,
    ).to(device)


def build_symbolic_dataframe(model, loader, split: str, use_reservoir: bool) -> tuple[pd.DataFrame, Dict[str, float]]:
    model.eval()
    reservoir = make_reservoir() if use_reservoir else None
    rows = []

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            _, latent = model(images)

            initial_state = adaptive_binarize(latent, cfg.reservoir_target_density)
            reservoir_diag = None
            if reservoir is not None:
                reservoir.reset_state()
                symbolic_latent, reservoir_diag = reservoir(latent, reset_state=True, return_diagnostics=True)
            else:
                symbolic_latent = initial_state

            for i in range(symbolic_latent.size(0)):
                feats = compute_latent_features(symbolic_latent[i], masks[i])
                feats.update(compute_reservoir_transition_features(initial_state[i], symbolic_latent[i], reservoir_diag, i))
                feats.update(compute_hsv_color_concepts(batch['image_path'][i], batch['mask_path'][i]))
                feats.update({
                    'image_id': batch['image_id'][i],
                    'class_id': int(batch['class_id'][i]),
                    'label_bin': int(batch['label_bin'][i]),
                    'split': split,
                })
                rows.append(feats)

    df = pd.DataFrame(rows)
    summary = {'rows': int(len(df)), 'use_reservoir': bool(use_reservoir)}
    for col in RESERVOIR_FEATURE_COLS:
        if col in df.columns:
            summary[f'{col}_mean'] = float(df[col].mean())
            summary[f'{col}_std'] = float(df[col].std(ddof=0))

    print(
        f"{split}: rows={len(df)} "
        f"res_changed_mean={summary.get('res_changed_fraction_mean', 0.0):.4f} "
        f"res_initial_density={summary.get('res_initial_density_mean', 0.0):.4f} "
        f"res_final_density={summary.get('res_final_density_mean', 0.0):.4f}"
    )
    return df, summary


# --- Quantile binning (multi-bin predicates) ---

def build_quantile_map(train_df: pd.DataFrame, feature_cols: List[str], quantiles: Tuple[float, float, float, float, float]):
    qmap = {}
    for feat in feature_cols:
        vals = train_df[feat].to_numpy(dtype=float)
        qmap[feat] = np.nanquantile(vals, quantiles)
    return qmap


def apply_quantile_bins(df: pd.DataFrame, qmap: Dict[str, np.ndarray], keep_raw: List[str]):
    out = df.copy()
    bin_cols = {}
    for feat, q in qmap.items():
        q1, q2, q3, q4, q5 = q
        vals = out[feat]
        bin_cols[f'{feat}_q1'] = (vals <= q1).astype(np.int8)
        bin_cols[f'{feat}_q2'] = ((vals > q1) & (vals <= q2)).astype(np.int8)
        bin_cols[f'{feat}_q3'] = ((vals > q2) & (vals <= q3)).astype(np.int8)
        bin_cols[f'{feat}_q4'] = ((vals > q3) & (vals <= q4)).astype(np.int8)
        bin_cols[f'{feat}_q5'] = ((vals > q4) & (vals <= q5)).astype(np.int8)
        bin_cols[f'{feat}_q6'] = (vals > q5).astype(np.int8)

    if bin_cols:
        out = pd.concat([out, pd.DataFrame(bin_cols, index=out.index)], axis=1)

    for col in keep_raw:
        if col in out.columns:
            out[col] = out[col].astype(np.int8)
    return out.copy()


# --- Clinical predicate layer (ABCD-style + reservoir state cues) ---

def add_clinical_predicates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def _any(cols: List[str]) -> pd.Series:
        cols = [c for c in cols if c in out.columns]
        if not cols:
            return pd.Series(0, index=out.index, dtype=np.int8)
        acc = out[cols[0]].astype(int)
        for c in cols[1:]:
            acc = acc | out[c].astype(int)
        return acc.astype(np.int8)

    out['clin_asymmetry'] = _any([
        'symmetry_score_q1', 'symmetry_score_q2',
        'mask_asymmetry_q5', 'mask_asymmetry_q6',
        'mask_eccentricity_q5', 'mask_eccentricity_q6',
    ])
    out['clin_border_irregular'] = _any([
        'border_irregularity_q5', 'border_irregularity_q6',
        'mask_border_std_q5', 'mask_border_std_q6',
        'mask_compactness_q1', 'mask_compactness_q2',
        'mask_solidity_q1', 'mask_solidity_q2',
    ])
    out['clin_color_variegation'] = _any([
        'hue_entropy_q5', 'hue_entropy_q6',
        'hue_variety_q5', 'hue_variety_q6',
        'saturation_std_q5', 'saturation_std_q6',
        'value_std_q5', 'value_std_q6',
        'color_variegated', 'color_irregular',
    ])
    out['clin_large_lesion'] = _any(['region_area_q5', 'region_area_q6'])
    out['clin_dark'] = _any(['value_mean_q1', 'value_mean_q2', 'color_dark'])
    out['clin_reddish'] = _any(['red_ratio_q5', 'red_ratio_q6', 'color_reddish'])
    out['clin_bluish'] = _any(['blue_ratio_q5', 'blue_ratio_q6', 'color_bluish'])
    out['clin_texture_rough'] = _any(['texture_heterogeneity_q5', 'texture_heterogeneity_q6'])

    out['clin_reservoir_unstable'] = _any([
        'res_changed_fraction_q5', 'res_changed_fraction_q6',
        'res_mean_step_delta_q5', 'res_mean_step_delta_q6',
        'res_last_step_delta_q5', 'res_last_step_delta_q6',
    ])
    out['clin_reservoir_expanding'] = _any([
        'res_density_shift_q5', 'res_density_shift_q6',
        'res_on_transition_rate_q5', 'res_on_transition_rate_q6',
    ])
    out['clin_reservoir_eroding'] = _any([
        'res_density_shift_q1', 'res_density_shift_q2',
        'res_off_transition_rate_q5', 'res_off_transition_rate_q6',
    ])
    out['clin_reservoir_persistent'] = _any([
        'res_persistence_rate_q5', 'res_persistence_rate_q6',
        'res_changed_fraction_q1', 'res_changed_fraction_q2',
    ])

    return out


feature_cols = [
    'latent_density', 'area_ratio', 'symmetry_score', 'compactness',
    'channel_diversity', 'center_periphery_contrast', 'border_irregularity', 'texture_heterogeneity',
    'region_mean', 'background_mean', 'region_bg_contrast', 'region_area',
    'region_coherence', 'region_active_channels', 'mask_compactness',
    'mask_eccentricity', 'mask_solidity', 'mask_border_std', 'mask_asymmetry',
    'value_mean', 'saturation_mean', 'hue_variety', 'red_ratio', 'blue_ratio',
    'saturation_std', 'value_std', 'hue_entropy',
] + RESERVOIR_FEATURE_COLS

for b in range(16):
    feature_cols.append(f'hue_bin_{b}')

raw_predicates = ['color_dark', 'color_variegated', 'color_reddish', 'color_bluish', 'color_irregular']

print('Symbolic output will be saved at:', cfg.symbol_out_dir)
print('Reservoir enabled:', cfg.use_reservoir)

split_summaries: Dict[str, Dict[str, float]] = {}
train_df, split_summaries['train'] = build_symbolic_dataframe(model, loaders['train'], 'train', cfg.use_reservoir)
val_df, split_summaries['val'] = build_symbolic_dataframe(model, loaders['val'], 'val', cfg.use_reservoir)
test_df, split_summaries['test'] = build_symbolic_dataframe(model, loaders['test'], 'test', cfg.use_reservoir)

missing_features = [c for c in feature_cols if c not in train_df.columns]
assert not missing_features, f'Missing feature columns: {missing_features}'

qmap = build_quantile_map(train_df, feature_cols, cfg.predicate_quantiles)

train_pred = add_clinical_predicates(apply_quantile_bins(train_df, qmap, raw_predicates))
val_pred = add_clinical_predicates(apply_quantile_bins(val_df, qmap, raw_predicates))
test_pred = add_clinical_predicates(apply_quantile_bins(test_df, qmap, raw_predicates))

pred_cols = [c for c in train_pred.columns if c not in ['image_id', 'class_id', 'label_bin', 'split']]
rates = train_pred[pred_cols].mean()
keep_cols = [c for c in pred_cols if cfg.predicate_min_rate < rates[c] < cfg.predicate_max_rate]

dropped = len(pred_cols) - len(keep_cols)
print(f'Dropping {dropped} always-on/off predicates. Keeping {len(keep_cols)} predicates.')

train_pred = train_pred[['image_id', 'class_id', 'label_bin', 'split'] + keep_cols]
val_pred = val_pred[['image_id', 'class_id', 'label_bin', 'split'] + keep_cols]
test_pred = test_pred[['image_id', 'class_id', 'label_bin', 'split'] + keep_cols]

train_df.to_csv(cfg.symbol_out_dir / f'seed_{cfg.run_seed}_features_train.csv', index=False)
val_df.to_csv(cfg.symbol_out_dir / f'seed_{cfg.run_seed}_features_val.csv', index=False)
test_df.to_csv(cfg.symbol_out_dir / f'seed_{cfg.run_seed}_features_test.csv', index=False)

train_pred.to_csv(cfg.symbol_out_dir / f'seed_{cfg.run_seed}_predicates_train.csv', index=False)
val_pred.to_csv(cfg.symbol_out_dir / f'seed_{cfg.run_seed}_predicates_val.csv', index=False)
test_pred.to_csv(cfg.symbol_out_dir / f'seed_{cfg.run_seed}_predicates_test.csv', index=False)

qmap_json = {k: [float(x) for x in v] for k, v in qmap.items()}
threshold_report = {
    'quantiles': qmap_json,
    'predicate_quantiles': list(cfg.predicate_quantiles),
    'feature_cols': feature_cols,
    'raw_predicates': raw_predicates,
    'kept_predicates': keep_cols,
    'dropped_predicate_count': int(dropped),
}
with (cfg.symbol_out_dir / f'seed_{cfg.run_seed}_feature_thresholds.json').open('w', encoding='utf-8') as f:
    pyjson.dump(threshold_report, f, indent=2)

reservoir_report = {
    'seed': cfg.run_seed,
    'variant_name': cfg.variant_name,
    'source_model_variant_name': cfg.source_model_variant_name,
    'use_reservoir': cfg.use_reservoir,
    'source_checkpoint': str(ckpt) if 'ckpt' in globals() else str(cfg.out_ckpt_best),
    'reservoir_config': {
        'layers': cfg.reservoir_layers,
        'steps': cfg.reservoir_steps,
        'target_density': cfg.reservoir_target_density,
        'state_coupling': cfg.reservoir_state_coupling,
        'input_coupling': cfg.reservoir_input_coupling,
        'leak': cfg.reservoir_leak,
        'bias': cfg.reservoir_bias,
        'until_converged': cfg.reservoir_until_converged,
        'stable_tol': cfg.reservoir_stable_tol,
    },
    'split_summaries': split_summaries,
}
with (cfg.symbol_out_dir / f'seed_{cfg.run_seed}_reservoir_diagnostics.json').open('w', encoding='utf-8') as f:
    pyjson.dump(reservoir_report, f, indent=2)

print('Saved symbolic features and predicates to:', cfg.symbol_out_dir)
print('Saved reservoir diagnostics:', cfg.symbol_out_dir / f'seed_{cfg.run_seed}_reservoir_diagnostics.json')


Symbolic output will be saved at: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Seed42_Feat
Reservoir enabled: False
train: rows=10784 res_changed_mean=0.0000 res_initial_density=0.1501 res_final_density=0.1501
val: rows=2312 res_changed_mean=0.0000 res_initial_density=0.1501 res_final_density=0.1501
test: rows=2312 res_changed_mean=0.0000 res_initial_density=0.1501 res_final_density=0.1501
Dropping 125 always-on/off predicates. Keeping 270 predicates.
Saved symbolic features and predicates to: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Seed42_Feat
Saved reservoir diagnostics: C:\Users\Admin\Desktop\Desktop\Uniud-Magistrale\SecondYear\LargeProject\Progetto-Medico\Github\Results\NoReservoir\results_seed42\Seed42_Feat\seed_42_reservoir_diagnostics.json
